# Content Based Recommendation System

In [147]:
import numpy as np
import pandas as pd

In [149]:
movies = pd.read_csv("../data/tmdb_5000_movies.csv")
credits = pd.read_csv("../data/tmdb_5000_credits.csv")

In [150]:
movies = movies.merge(credits, on = 'title')

In [151]:
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


## Selecting Important Columns for Recommendation System

For building our content-based movie recommendation system, we do not need every column from the dataset.  
We only select the columns that help us understand the movie content and similarity between movies.

### Selected Columns

1. **id**
   - Used as a unique identifier for each movie.

2. **keywords**
   - Contains important themes and topics related to the movie.
   - Helps in understanding the movie context.

3. **title**
   - Stores the movie name.
   - Useful for displaying recommendations.

4. **overview**
   - Gives a short description or storyline of the movie.
   - One of the most important features for content-based recommendation.

5. **release_date**
   - Some users prefer movies from a specific era (e.g., 1990s or classic old movies).
   - Although this can be an important factor, we are not considering it in this project because it is numerical/date-based data.

6. **cast**
   - Many users watch movies based on their favorite actors or actresses.
   - Therefore, cast information becomes an important recommendation feature.

7. **crew**
   - Includes director, writer, producer, etc.
   - Directors often have a unique filmmaking style, so this information helps improve recommendations.

In [153]:

movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]


## Preprocessing

In [155]:
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [156]:
#so just drop those 3 rows of overview
movies.dropna(inplace = True) 

In [157]:
movies.isnull().sum()

movie_id    0
title       0
overview    0
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [158]:
movies.duplicated().sum()

np.int64(0)

### Since duplicated().sum() gives zero as a value means there is no any duplicate present in our dataset

In [160]:
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [161]:
obj = '[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

## Understanding `ast.literal_eval()` in Our Recommendation System

While working with the movie dataset, we observed that some columns such as:

- `genres`
- `keywords`
- `cast`
- `crew`

do not contain normal Python lists directly.  
Instead, their values are stored as **strings that look like Python objects**.

Example:

```python
"[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}]"
```

Although this looks like a list of dictionaries, Python actually treats it as a **string**.

---

## Problem

Since the value is stored as a string, we cannot directly loop through it like a normal list.

For example:

```python
type(obj)
```

Output:

```python
str
```

This means Python considers it plain text, not an actual list.

---

## Solution: `ast.literal_eval()`

To solve this problem, we use:

```python
import ast
```

and then:

```python
ast.literal_eval(obj)
```

The `literal_eval()` function converts a string representation of a Python object into an actual Python object safely.

---

## Example

### Before Conversion

```python
obj = "[{'id': 28, 'name': 'Action'}]"
```

Type:

```python
type(obj)
```

Output:

```python
str
```

---

### After Conversion

```python
ast.literal_eval(obj)
```

Output:

```python
[{'id': 28, 'name': 'Action'}]
```

Now the type becomes:

```python
list
```

This allows us to iterate through the data properly.

---

## Why We Need This in Our Project

For our content-based recommendation system, we only need meaningful textual information such as:

- Action
- Adventure
- Fantasy
- Science Fiction

We do not need IDs.

So we extract only the `name` values from the dictionaries.

---

## Code Used

```python
def convert(obj):
    L = []
    
    for i in ast.literal_eval(obj):
        L.append(i['name'])
        
    return L
```

---

## Explanation of the Code

### Step 1

```python
L = []
```

Creates an empty list to store genre names.

---

### Step 2

```python
for i in ast.literal_eval(obj):
```

- Converts the string into an actual Python list.
- Iterates through each dictionary.

Example iteration:

```python
{'id': 28, 'name': 'Action'}
```

---

### Step 3

```python
L.append(i['name'])
```

Extracts only the value associated with the `name` key.

Example:

```python
i['name']
```

Output:

```python
'Action'
```

---

## Final Output

```python
['Action', 'Adventure', 'Fantasy', 'Science Fiction']
```

This cleaned textual data becomes part of our `tags` column, which is later used for:

- text vectorization
- cosine similarity
- movie recommendations

---

## Importance in the Recommendation System

This preprocessing step is very important because machine learning models cannot understand complex nested string structures directly.

By converting and cleaning the data:

- we extract meaningful features,
- simplify the dataset,
- and prepare the data for vectorization and similarity calculation.

This is an important part of the data preprocessing and feature engineering pipeline in machine learning projects.

In [163]:
import ast
ast.literal_eval(obj)

[{'id': 28, 'name': 'Action'},
 {'id': 12, 'name': 'Adventure'},
 {'id': 14, 'name': 'Fantasy'},
 {'id': 878, 'name': 'Science Fiction'}]

In [164]:
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [166]:
movies['genres'] = movies['genres'].apply(convert)

In [172]:
#similarly for keywords
movies['keywords'] = movies['keywords'].apply(convert)

In [173]:
def convert3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

In [174]:
movies['cast'] = movies['cast'].apply(convert3)

In [175]:
def fetch_director(obj):
    L= []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

In [176]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [177]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [179]:
# now we have to remove the space between the names like take Sam Worthington -> SamWorthington
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])


In [186]:
#tags 
movies['tags'] = movies['overview'] + movies['genres'] +  movies['keywords'] + movies['cast'] + movies['crew']

In [189]:
movies.head(2)['tags']

0    [In, the, 22nd, century,, a, paraplegic, Marin...
1    [Captain, Barbossa,, long, believed, to, be, d...
Name: tags, dtype: object

In [190]:
new_df = movies[['movie_id','title','tags']]

In [191]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [201]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

In [205]:
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower()) # converted into lower case

In [207]:
new_df['tags'][0]

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron'

In [209]:
new_df['tags'][1]

"captain barbossa, long believed to be dead, has come back to life and is headed to the edge of the earth with will turner and elizabeth swann. but nothing is quite as it seems. adventure fantasy action ocean drugabuse exoticisland eastindiatradingcompany loveofone'slife traitor shipwreck strongwoman ship alliance calypso afterlife fighter pirate swashbuckler aftercreditsstinger johnnydepp orlandobloom keiraknightley goreverbinski"

In [211]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


---
# Vectorization

In [215]:
from sklearn.feature_extraction.text import CountVectorizer

In [217]:
cv = CountVectorizer(max_features = 5000, stop_words = 'english')

#### What stop_words='english' Means

Removes common English words like:
the,is,and,are and many more like this ....

Because they don’t help recommendation quality much.
This is basic NLP preprocessing.

In [220]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [254]:
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(4806, 5000))

In [224]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      shape=(5000,), dtype=object)

In [226]:
import nltk

In [228]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [230]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [232]:
ps.stem('danced')

'danc'

In [234]:
stem(new_df['tags'][0])

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d samworthington zoesaldana sigourneyweav jamescameron'

In [236]:
new_df['tags'] = new_df['tags'].apply(stem)

In [237]:
new_df.head(2)

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."


In [244]:
from sklearn.metrics.pairwise import cosine_similarity

In [246]:
similarity = cosine_similarity(vectors)

In [306]:
sorted(list(enumerate(similarity[0].tolist())),reverse = True, key = lambda x:x[1])[1:6]

[(539, 0.25038669783359574),
 (1192, 0.24779731389167606),
 (507, 0.24283093212859141),
 (260, 0.2409900932515112),
 (1214, 0.23939494881986934)]

In [324]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)),reverse=True, key=lambda x:x[1])[1:6]

    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [326]:
recommend('Avatar')

Titan A.E.
Small Soldiers
Independence Day
Ender's Game
Aliens vs Predator: Requiem
